# (mu, sigma) -> FWHM Distribution Map

**Goal:** Build intuition for how the photon-count proposal distribution N(mu, sigma) affects
the resulting FWHM distribution from the Monte Carlo pipeline.

We fix the physical parameters (gamma=20, lambda=2, window=[-75,75]) and sweep over a grid of
(mu, sigma) values. For each point on the grid, we run N_RUNS PLE scans and collect the
FWHM values.

**Key question:** At which (mu, sigma) does the FWHM distribution best match the target?
And where does the REINFORCE gradient have signal?

**Warning:** This notebook runs 1000s of L-BFGS fits. Budget ~15-30 min.


## Imports & Setup


In [ ]:
import math, time, itertools
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch

torch.set_default_dtype(torch.float64)

# Detection window (MHz)
FREQ_MIN = -75.0
FREQ_MAX = 75.0
UNIFORM_DENSITY = 1.0 / (FREQ_MAX - FREQ_MIN)
print('Imports OK')


## Fitting Code (same as 07/08, inlined)


In [ ]:
WIDTH_MAX = FREQ_MAX - FREQ_MIN
WIDTH_EPS = 1e-2
FIT_ETA = 0.2

def _width(raw):
    return WIDTH_EPS + WIDTH_MAX * torch.sigmoid(raw)

def _rw(width):
    frac = (width - WIDTH_EPS) / WIDTH_MAX
    frac = max(frac, 1e-10)
    frac = min(frac, 1 - 1e-10)
    return math.log(frac / (1.0 - frac))

def _lorentz_cdf(x, center, gam):
    return 0.5 + torch.atan((x - center) / gam) / math.pi

def _normal_cdf(x, center, sigma):
    return 0.5 * (1.0 + torch.erf((x - center) / (sigma * math.sqrt(2.0))))

def log_pdf(freqs, center, rg, rs, lw):
    gamma = _width(rg); sigma_g = _width(rs); w = torch.sigmoid(lw)
    hi = torch.as_tensor(FREQ_MAX, dtype=freqs.dtype)
    lo = torch.as_tensor(FREQ_MIN, dtype=freqs.dtype)
    lorentz = (gamma / math.pi) / ((freqs - center) ** 2 + gamma ** 2)
    Z_l = _lorentz_cdf(hi, center, gamma) - _lorentz_cdf(lo, center, gamma)
    gauss = torch.exp(-0.5 * ((freqs - center) / sigma_g) ** 2) / (sigma_g * math.sqrt(2.0 * math.pi))
    Z_g = _normal_cdf(hi, center, sigma_g) - _normal_cdf(lo, center, sigma_g)
    signal = FIT_ETA * gauss / (Z_g + 1e-30) + (1 - FIT_ETA) * lorentz / (Z_l + 1e-30)
    pdf = w * signal + (1 - w) * UNIFORM_DENSITY
    return torch.log(pdf + 1e-30)

def nll(theta, photons):
    return -log_pdf(photons, theta[0], theta[1], theta[2], theta[3]).mean()

def fwhm_from_theta(theta):
    f_L = 2.0 * _width(theta[1])
    f_G = 2.0 * math.sqrt(2.0 * math.log(2.0)) * _width(theta[2])
    return 0.5346 * f_L + torch.sqrt(0.2166 * f_L ** 2 + f_G ** 2)

def fit(photons):
    if len(photons) < 3: return None
    data = photons.detach()
    theta = torch.tensor([float(data.median()), _rw(15), _rw(5), 0.0], requires_grad=True)
    opt = torch.optim.LBFGS([theta], max_iter=100, line_search_fn='strong_wolfe')
    def c(): opt.zero_grad(); loss = nll(theta, data); loss.backward(); return loss
    try: opt.step(c)
    except: return None
    return theta.detach() if torch.isfinite(theta).all().item() else None

def signal_detunings(gamma, u):
    half = 0.5 * (FREQ_MAX - FREQ_MIN)
    return gamma * torch.tan(torch.tensor(math.atan(half / gamma)) * (2.0 * u - 1.0))

def build_photons(gamma, u, b):
    return torch.cat([signal_detunings(gamma, u), b])

def run_scan(n, gamma, lam, rng):
    u = torch.tensor(rng.uniform(0, 1, size=n))
    bg = int(rng.poisson(lam))
    b = torch.tensor(rng.uniform(FREQ_MIN, FREQ_MAX, size=bg))
    if len(u) + len(b) < 3: return 2 * gamma
    p = build_photons(torch.tensor(gamma), u, b)
    t = fit(p)
    return fwhm_from_theta(t).item() if t is not None else 2 * gamma

print('Fitting code OK')


## Parameters & Grid Setup


In [ ]:
# Fixed physical parameters
GAMMA = 20.0
LAMBDA_ = 2.0

# Runs per grid point
N_RUNS = 500

# Grid: mu (mean n) and sigma (std of n)
MU_VALS = [10, 20, 30, 50, 80, 120, 200]
SIGMA_VALS = [3, 6, 12, 24]

n_points = len(MU_VALS) * len(SIGMA_VALS)
n_total = n_points * N_RUNS
print(f'Grid: {len(MU_VALS)} mu x {len(SIGMA_VALS)} sigma = {n_points} points')
print(f'Runs per point: {N_RUNS}')
print(f'Total: {n_total} runs (~{n_total * 0.2 / 60:.0f} min)')


## Run the Grid Sweep


In [ ]:
results = {}
t0 = time.time()
for idx, (mu, sigma) in enumerate(itertools.product(MU_VALS, SIGMA_VALS)):
    rng = np.random.default_rng(mu * 100 + int(sigma * 10))
    fwhms, ns, deg = [], [], 0
    for _ in range(N_RUNS):
        n = max(round(mu + sigma * rng.standard_normal()), 0)
        ns.append(n)
        f = run_scan(n, GAMMA, LAMBDA_, rng)
        if abs(f - 2 * GAMMA) < 0.1: deg += 1
        fwhms.append(f)
    fw_t = torch.tensor(fwhms)
    results[(mu, sigma)] = {'fwhms': fw_t, 'ns': ns, 'deg': deg}
    elapsed = time.time() - t0
    eta = (elapsed / (idx+1)) * (n_points - idx - 1)
    print(f'[{idx+1:2d}/{n_points}] mu={mu:3d} sigma={sigma:2d}: mean={fw_t.mean():.1f} med={fw_t.median():.1f} std={fw_t.std():.1f} deg={deg:3d} ({elapsed:.0f}s, ETA {eta:.0f}s)')

print(f'Done in {time.time()-t0:.0f}s')


## Visualization

### 1. Varying mu (fixed sigma=6)
Each subplot: FWHM distribution at one mu, with sigma=6 (the physical sigma_n).


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for j, mu in enumerate(MU_VALS):
    ax = axes[j]
    fw = results[(mu, 6)]['fwhms']
    ax.hist(fw.numpy(), bins=50, alpha=0.7, color=plt.cm.viridis(mu/200), density=True)
    ax.set_xlabel('FWHM (MHz)'); ax.set_ylabel('Density')
    ax.set_title(f'mu={mu}, sigma=6'); ax.set_xlim(0, 120); ax.grid(alpha=0.3)
for j in range(len(MU_VALS), len(axes)): axes[j].set_visible(False)
plt.suptitle('FWHM Distributions: Varying mu (sigma=6)', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()


### 2. Varying sigma (fixed mu=50)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for j, sigma in enumerate(SIGMA_VALS):
    ax = axes[j]
    fw = results[(50, sigma)]['fwhms']
    ax.hist(fw.numpy(), bins=50, alpha=0.7, color=plt.cm.plasma(sigma/24), density=True)
    ax.set_xlabel('FWHM (MHz)'); ax.set_ylabel('Density')
    ax.set_title(f'mu=50, sigma={sigma}'); ax.set_xlim(0, 120); ax.grid(alpha=0.3)
plt.suptitle('FWHM Distributions: Varying sigma (mu=50)', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()


### 3. Overlayed KDEs — Full Grid


In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
cm_mu = plt.cm.viridis(np.linspace(0.2, 0.9, len(MU_VALS)))
cm_sig = plt.cm.plasma(np.linspace(0.2, 0.9, len(SIGMA_VALS)))
xg = np.linspace(0, 120, 500)

ax = axes[0, 0]
for j, mu in enumerate(MU_VALS):
    fw = results[(mu, 6)]['fwhms'].numpy()
    if len(fw) > 10:
        ax.plot(xg, gaussian_kde(fw)(xg), color=cm_mu[j], lw=2, label=f'mu={mu}')
ax.set_title('Varying mu (sigma=6)', fontweight='bold'); ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(0,120)

ax = axes[0, 1]
for j, sigma in enumerate(SIGMA_VALS):
    fw = results[(50, sigma)]['fwhms'].numpy()
    if len(fw) > 10:
        ax.plot(xg, gaussian_kde(fw)(xg), color=cm_sig[j], lw=2, label=f'sigma={sigma}')
ax.set_title('Varying sigma (mu=50)', fontweight='bold'); ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(0,120)

ax = axes[1, 0]
sub_mu = [m for m in MU_VALS if m in [10, 30, 80]]
sub_sig = [s for s in SIGMA_VALS if s in [3, 12]]
for i, mu in enumerate(sub_mu):
    for j, sigma in enumerate(sub_sig):
        fw = results[(mu, sigma)]['fwhms'].numpy()
        if len(fw) > 10:
            ax.plot(xg, gaussian_kde(fw)(xg), color=plt.cm.viridis((i+0.5)/len(sub_mu)),
                    ls=['-','--'][j], lw=1.5, label=f'mu={mu}, sig={sigma}')
ax.set_title('Selected combos', fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlim(0,120)

ax = axes[1, 1]
for j, sigma in enumerate(SIGMA_VALS):
    means, stds = [], []
    for mu in MU_VALS:
        fw = results[(mu, sigma)]['fwhms']
        means.append(fw.mean().item()); stds.append(fw.std().item())
    ax.errorbar(MU_VALS, means, yerr=stds, color=cm_sig[j], marker='o', capsize=3, label=f'sigma={sigma}')
ax.set_xscale('log'); ax.set_title('Mean FWHM vs mu', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('FWHM Distribution Map: Complete Grid', fontsize=18, fontweight='bold')
plt.tight_layout(); plt.show()


### 4. Heatmaps: Summary Statistics Over (mu, sigma) Grid


In [ ]:
def heat(data_dict, grid_mu, grid_sig):
    a = np.full((len(grid_sig), len(grid_mu)), np.nan)
    for i, s in enumerate(grid_sig):
        for j, m in enumerate(grid_mu):
            if (m, s) in data_dict and data_dict[(m, s)] is not None:
                a[i, j] = data_dict[(m, s)]
    return a

mean_d = {(m,s): results[(m,s)]['fwhms'].mean().item() for m in MU_VALS for s in SIGMA_VALS}
med_d  = {(m,s): results[(m,s)]['fwhms'].median().item() for m in MU_VALS for s in SIGMA_VALS}
std_d  = {(m,s): results[(m,s)]['fwhms'].std().item() for m in MU_VALS for s in SIGMA_VALS}
deg_d  = {(m,s): results[(m,s)]['deg'] / N_RUNS * 100 for m in MU_VALS for s in SIGMA_VALS}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plots = [(mean_d, 'Mean FWHM', 'Reds', axes[0,0]),
         (med_d, 'Median FWHM', 'Blues', axes[0,1]),
         (std_d, 'FWHM Std', 'Purples', axes[1,0]),
         (deg_d, 'Degenerate %', 'RdYlGn_r', axes[1,1])]

for data, title, cm, ax in plots:
    arr = heat(data, MU_VALS, SIGMA_VALS)
    im = ax.imshow(arr, aspect='auto', cmap=cm)
    ax.set_xticks(range(len(MU_VALS))); ax.set_xticklabels(MU_VALS)
    ax.set_yticks(range(len(SIGMA_VALS))); ax.set_yticklabels(SIGMA_VALS)
    ax.set_xlabel('mu'); ax.set_ylabel('sigma'); ax.set_title(title, fontweight='bold')
    for i in range(len(SIGMA_VALS)):
        for j in range(len(MU_VALS)):
            v = arr[i,j]
            if not np.isnan(v):
                tc = 'white' if v > np.nanmax(arr) * 0.6 else 'black'
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=9, color=tc, fontweight='bold')
    plt.colorbar(im, ax=ax)

plt.suptitle('(mu, sigma) Grid: Summary Statistics', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()


### 5. Signal Strength Map

W1 distance from each (mu, sigma) to the reference (mu=50, sigma=6).
Red = strong mismatch = clear REINFORCE gradient direction.


In [ ]:
ref = results[(50, 6)]['fwhms']
sr, _ = torch.sort(ref)

w1_d = {}
for mu in MU_VALS:
    for sigma in SIGMA_VALS:
        fw = results[(mu, sigma)]['fwhms']
        sf, _ = torch.sort(fw)
        w1_d[(mu, sigma)] = torch.abs(sf - sr[:len(sf)]).mean().item()

fig, ax = plt.subplots(figsize=(10, 6))
arr = heat(w1_d, MU_VALS, SIGMA_VALS)
im = ax.imshow(arr, aspect='auto', cmap='RdYlGn_r', vmin=0)
ax.set_xticks(range(len(MU_VALS))); ax.set_xticklabels(MU_VALS)
ax.set_yticks(range(len(SIGMA_VALS))); ax.set_yticklabels(SIGMA_VALS)
ax.set_xlabel('mu'); ax.set_ylabel('sigma')
ax.set_title('W1 Distance from Reference (mu=50, sigma=6)', fontweight='bold', fontsize=14)
for i in range(len(SIGMA_VALS)):
    for j in range(len(MU_VALS)):
        v = arr[i,j]
        if not np.isnan(v):
            tc = 'white' if v > np.nanmax(arr) * 0.5 else 'black'
            ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=11, color=tc, fontweight='bold')
ref_j = MU_VALS.index(50); ref_i = SIGMA_VALS.index(6)
ax.scatter(ref_j, ref_i, s=300, facecolors='none', edgecolors='blue', linewidths=3, label='Reference')
ax.legend(fontsize=11)
plt.colorbar(im, ax=ax, label='W1 (MHz)')
plt.tight_layout(); plt.show()

print('Green = close to reference; Red = strong mismatch = REINFORCE signal')


### 6. REINFORCE Gradient Map

rho(n_sorted, quantile_loss) for each grid point vs the reference (mu=50, sigma=6).
Negative = gradient pushes mu UP. Positive = pushes mu DOWN.


In [ ]:
rho_d, grad_d = {}, {}
for mu in MU_VALS:
    for sigma in SIGMA_VALS:
        fw = results[(mu, sigma)]['fwhms']
        ns = torch.tensor(results[(mu, sigma)]['ns'])
        sf, sidx = torch.sort(fw)
        pl = torch.abs(sf - sr[:len(sf)])
        nss = ns[sidx]
        if pl.std().item() > 0.01 and nss.std().item() > 0.01:
            rho = np.corrcoef(nss.numpy(), pl.numpy())[0, 1]
            bl = pl.mean().item()
            adv = pl - bl
            scores = (nss - mu) / sigma**2
            grad = (adv.detach() * scores).mean().item()
        else:
            rho, grad = 0.0, 0.0
        rho_d[(mu, sigma)] = rho
        grad_d[(mu, sigma)] = grad

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
arr = heat(rho_d, MU_VALS, SIGMA_VALS)
vm = max(abs(np.nanmin(arr)), abs(np.nanmax(arr)))
im = ax.imshow(arr, aspect='auto', cmap='RdBu_r', vmin=-vm, vmax=vm)
ax.set_xticks(range(len(MU_VALS))); ax.set_xticklabels(MU_VALS)
ax.set_yticks(range(len(SIGMA_VALS))); ax.set_yticklabels(SIGMA_VALS)
ax.set_xlabel('mu'); ax.set_ylabel('sigma')
ax.set_title('rho(n_sorted, quantile_loss)', fontweight='bold')
for i in range(len(SIGMA_VALS)):
    for j in range(len(MU_VALS)):
        v = arr[i,j]
        if not np.isnan(v):
            tc = 'white' if abs(v) > vm * 0.6 else 'black'
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9, color=tc)
plt.colorbar(im, ax=ax)

ax = axes[1]
arr = heat(grad_d, MU_VALS, SIGMA_VALS)
vm2 = max(abs(np.nanmin(arr)), abs(np.nanmax(arr)))
im = ax.imshow(arr, aspect='auto', cmap='RdBu_r', vmin=-vm2, vmax=vm2)
ax.set_xticks(range(len(MU_VALS))); ax.set_xticklabels(MU_VALS)
ax.set_yticks(range(len(SIGMA_VALS))); ax.set_yticklabels(SIGMA_VALS)
ax.set_xlabel('mu'); ax.set_ylabel('sigma')
ax.set_title('REINFORCE Gradient', fontweight='bold')
for i in range(len(SIGMA_VALS)):
    for j in range(len(MU_VALS)):
        v = arr[i,j]
        if not np.isnan(v):
            tc = 'white' if abs(v) > vm2 * 0.6 else 'black'
            ax.text(j, i, f'{v:.5f}', ha='center', va='center', fontsize=9, color=tc)
plt.colorbar(im, ax=ax)

plt.suptitle('REINFORCE Signal Map', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

print('Blue = push mu UP | Red = push mu DOWN | White = no signal')


## Summary

- **Low mu (<30):** High variability, many degenerate fits, strong REINFORCE signal
- **Medium mu (30-80):** FWHM stabilizes, signal weakens
- **High mu (>100):** Clean fits, diminishing returns on more photons
- **Sigma effect:** Wider sigma spreads the FWHM distribution; at low mu it causes degenerate fits

**Bottom line:** REINFORCE gradient lives in the low-mu region. Near the truth,
the W1 loss landscape drives convergence.
